In [0]:
# ============================================================
# Config
# ============================================================
from pyspark.sql.functions import (
    col, countDistinct, sum, avg, round, 
    month, year, quarter, dayofweek, 
    monthname, current_timestamp, lit,
    row_number, dense_rank
)
from pyspark.sql.window import Window

SILVER_TABLE = "invoices.silver.invoices_clean"
GOLD_SCHEMA  = "invoices.gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")

# Read Silver — only current records
df_silver = spark.table(SILVER_TABLE)

print(f"Silver rows available: {df_silver.count()}")
display(df_silver.limit(3))

In [0]:
# ============================================================
# dim_client
# ============================================================

dim_client = (
    df_silver
    .select("client_name", "client_address")
    .distinct()
    .withColumn("client_key", dense_rank().over(
        Window.orderBy("client_name")
    ))
    .select("client_key", "client_name", "client_address")
)

(
    dim_client.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("invoices.gold.dim_client")
)

print(f"dim_client: {dim_client.count()} rows")
display(dim_client.limit(5))

In [0]:
# ============================================================
# dim_seller
# ============================================================

dim_seller = (
    df_silver
    .select("seller_name", "seller_address")
    .distinct()
    .withColumn("seller_key", dense_rank().over(
        Window.orderBy("seller_name")
    ))
    .select("seller_key", "seller_name", "seller_address")
)

(
    dim_seller.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("invoices.gold.dim_seller")
)

print(f"dim_seller: {dim_seller.count()} rows")
display(dim_seller.limit(5))

In [0]:
# ============================================================
# dim_date
# ============================================================
from pyspark.sql.functions import (
    year, month, quarter, dayofweek, 
    date_format, weekofyear
)

dim_date = (
    df_silver
    .select("invoice_date")
    .filter(col("invoice_date").isNotNull())
    .distinct()
    .withColumn("year",         year(col("invoice_date")))
    .withColumn("quarter",      quarter(col("invoice_date")))
    .withColumn("month",        month(col("invoice_date")))
    .withColumn("month_name",   date_format(col("invoice_date"), "MMMM"))
    .withColumn("week_of_year", weekofyear(col("invoice_date")))
    .withColumn("day_of_week",  dayofweek(col("invoice_date")))
    .withColumn("day_name",     date_format(col("invoice_date"), "EEEE"))
    .withColumnRenamed("invoice_date", "date_key")
)

(
    dim_date.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("invoices.gold.dim_date")
)

print(f"dim_date: {dim_date.count()} rows")
display(dim_date.limit(5))

In [0]:
# ============================================================
# fact_invoices
# ============================================================

fact_invoices = (
    df_silver
    .join(dim_client,  on="client_name",  how="left")
    .join(dim_seller,  on="seller_name",  how="left")
    .join(dim_date,    df_silver.invoice_date == dim_date.date_key, how="left")
    .select(
        # Keys
        col("invoice_number"),
        col("client_key"),
        col("seller_key"),
        col("invoice_date").alias("date_key"),

        # Invoice header
        col("invoice_due_date"),
        col("file_name"),
        col("batch_id"),

        # Line item measures
        col("item_description"),
        col("item_quantity"),
        col("item_total_price"),

        # Invoice level measures
        col("tax"),
        col("discount"),
        col("total"),

        # Payment info
        col("payment_method"),
        col("bank_name"),
        col("account_number"),

        # Audit
        col("ingestion_date"),
        col("cleaned_at")
    )
)

(
    fact_invoices.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("ingestion_date")
    .saveAsTable("invoices.gold.fact_invoices")
)

print(f"fact_invoices: {fact_invoices.count()} rows")
display(fact_invoices.limit(5))